# 00 · One-click setup: database + API, all from Jupyter
Run this notebook **top to bottom first**, then run `01_collect_clean_load.ipynb`. No terminal needed for the backend.

| Step | What it does |
|---|---|
| 1 | Choose `DB_MODE`: `"mysql"` (what the assignment wants) or `"sqlite"` (zero-install fallback, works instantly) |
| 2 | Installs the Python packages into *this* kernel |
| 3 | (MySQL mode) creates the database, user and the `listing_master` table from `database/schema.sql` |
| 4 | Writes `backend/.env` |
| 5 | Starts the FastAPI server in the background and waits until `/health` is OK |

The `.sql` dump written by notebook 01 is **MySQL syntax in both modes**, so you can submit it either way.

## 1 · Choose your database

In [4]:
DB_MODE = "mysql"            # "mysql" or "sqlite"  <- switch to "sqlite" if MySQL isn't installed yet
MYSQL_HOST, MYSQL_PORT = "127.0.0.1", 3306
MYSQL_ROOT_USER, MYSQL_ROOT_PASSWORD = "root", ""     # <- your MySQL root password (set during install)
APP_DB, APP_USER, APP_PASS = "business_listings", "bl_user", "bl_pass"
API_PORT = 8000

## 2 · Install packages into this kernel

In [5]:
import sys, subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "fastapi", "uvicorn[standard]", "sqlalchemy>=2",
                       "pymysql", "cryptography", "pydantic>=2", "pydantic-settings", "python-dotenv",
                       "pandas", "requests", "matplotlib"])
print("packages ready")

packages ready


## 3 · Create the database and table (MySQL mode)

In [6]:
import re
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
assert (ROOT / "backend").exists(), f"Run from the notebooks/ folder (looked in {ROOT})"

if DB_MODE == "mysql":
    import pymysql
    try:
        conn = pymysql.connect(host=MYSQL_HOST, port=MYSQL_PORT, user=MYSQL_ROOT_USER, password=MYSQL_ROOT_PASSWORD, autocommit=True)
    except pymysql.err.OperationalError as e:
        raise SystemExit(f"Cannot connect to MySQL: {e}\n-> Is the server running? Is MYSQL_ROOT_PASSWORD right? "
                         "Or set DB_MODE = 'sqlite' above to continue without MySQL.")
    cur = conn.cursor()
    cur.execute(f"CREATE DATABASE IF NOT EXISTS {APP_DB} CHARACTER SET utf8mb4")
    cur.execute(f"CREATE USER IF NOT EXISTS '{APP_USER}'@'%' IDENTIFIED BY '{APP_PASS}'")
    cur.execute(f"CREATE USER IF NOT EXISTS '{APP_USER}'@'localhost' IDENTIFIED BY '{APP_PASS}'")
    cur.execute(f"GRANT ALL ON {APP_DB}.* TO '{APP_USER}'@'%'"); cur.execute(f"GRANT ALL ON {APP_DB}.* TO '{APP_USER}'@'localhost'")
    cur.execute(f"USE {APP_DB}")
    sql = "\n".join(l for l in (ROOT / "database" / "schema.sql").read_text().splitlines() if not l.strip().startswith("--"))
    for stmt in filter(None, (s.strip() for s in sql.split(";"))):
        if not stmt.upper().startswith(("CREATE DATABASE", "USE")):
            cur.execute(stmt)
    cur.execute("SHOW TABLES"); print("MySQL ready. Tables:", cur.fetchall())
    DATABASE_URL = f"mysql+pymysql://{APP_USER}:{APP_PASS}@{MYSQL_HOST}:{MYSQL_PORT}/{APP_DB}"
else:
    DATABASE_URL = "sqlite:///./listings.db"          # file created inside backend/
    print("SQLite mode: table is created automatically when the API starts.")

MySQL ready. Tables: (('listing_master',),)


## 4 · Write `backend/.env` and start the API

In [7]:
(ROOT / "backend" / ".env").write_text(f"DATABASE_URL={DATABASE_URL}\nCORS_ORIGINS=http://localhost:5173,http://127.0.0.1:5173\n")

import time, requests, os
try:                                   # stop an older server from a previous run of this cell
    API_PROC.terminate(); API_PROC.wait(5)
except Exception:
    pass
log = open(ROOT / "backend" / "uvicorn.log", "w")
API_PROC = subprocess.Popen([sys.executable, "-m", "uvicorn", "app.main:app", "--port", str(API_PORT)],
                            cwd=ROOT / "backend", stdout=log, stderr=subprocess.STDOUT)
for _ in range(40):
    try:
        if requests.get(f"http://127.0.0.1:{API_PORT}/health", timeout=2).json()["status"] == "ok": break
    except Exception:
        time.sleep(0.5)
else:
    raise SystemExit("API did not start - open backend/uvicorn.log to see why (port busy? wrong DB password?)")
print(f"API is running ->  http://127.0.0.1:{API_PORT}/docs   (database: {DB_MODE})")

API is running ->  http://127.0.0.1:8000/docs   (database: mysql)


## ✅ Done - now open and run `01_collect_clean_load.ipynb`
Keep this kernel alive while you work (the API is a child process of it). When finished, run the cell below.

In [ ]:
# Run this only when you want to stop the API.
API_PROC.terminate(); print("API stopped")